# Molecular Docking

Il docking molecolare risponde a una domanda semplice: quanto bene si lega una piccola molecola (ligando) a una proteina target?
Il software esplora le possibili pose del ligando nel sito di legame e restituisce un punteggio energetico (in kcal/mol). Più negativo è il valore, più forte è il legame predetto.

Il workflow classico è:

- Hai una proteina con un sito di legame noto
- Hai una o più molecole candidate (ligandi)
- Il software (AutoDock Vina, nel nostro caso) cerca la posa di minima energia
- Analizzi i risultati


** Il nostro caso d'uso ** : ABL1 e Imatinib
Useremo ABL1, una proteina chinasi coinvolta nella leucemia mieloide cronica. È il target di Imatinib (Gleevec), uno dei farmaci oncologici più famosi della storia, il primo esempio di terapia mirata molecolare.

In [ ]:
! apt-get install -y autodock-vina openbabel
!pip install dockstring rdkit pandas matplotlib seaborn py3Dmol

In [ ]:
import dockstring
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw, AllChem
import pandas as pd
import py3Dmol

In [ ]:
print(f"Dockstring : {dockstring.__version__}")
print(f"Rdkit: {rdkit.__version__}")
print(f"py3Dmol: {dockstring.__version__}")

#Prima molecola: rappresentare Imatinib

In [ ]:
smile="Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1"

mol=Chem.MolFromSmiles(smile)

In [ ]:
# COME alternativa
import requests

mol_name="imatinib"

r=requests.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{mol_name}/property/IsomericSMILES/JSON")
data=r.json()



In [ ]:
print(data)

In [ ]:
smile_json=data["PropertyTable"]["Properties"][0]["SMILES"]
print(smile_json)

In [ ]:
smile==smile_json

I due SMILES sembrano diversi ma in realtà rappresentanto la stessa molecola. La stessa molecola può avere infinite rappresentazioni SMILES valide, a seconda da quale atomo parte la traversata del grafo molecolare.

Per dimostrare che sono la stessa molecola, usiamo
MolFromSmiles che è la funzione di rdkit.Chem che ci restituisce la forma canonica dello SMILE.

In [ ]:
mol_json=Chem.MolFromSmiles(smile_json)

In [ ]:
print(f"Manual_smile={Chem.MolToSmiles(mol)}")
print(f"GET smile = {Chem.MolToSmiles(mol_json)}")
Chem.MolToSmiles(mol)==Chem.MolToSmiles(mol_json)


Un altro approccio è con  InChI Key. Questo è un identificatore molecolare hash, completamente indipendente dallo SMILES. Due molecole identiche hanno sempre lo stesso InChI Key

In [ ]:
Chem.MolToInchi(mol)==Chem.MolToInchi(mol_json)

In [ ]:
# Visualizziamole
molecules=[mol,mol_json]
legend=["Manual SMILE","JSON SMILE"]
Draw.MolsToGridImage(molecules,legends=legend,molsPerRow=2,subImgSize=(250,250))

RDKit le sta disegnando con un layout 2D diverso perché parte da atomi diversi quando genera le coordinate.

Possiamo allineare le coordinate 2D di una molecola su quelle dell'altra, in modo da avere la stessa rappresentazione.

In [ ]:
AllChem.Compute2DCoords(mol)


In [ ]:
AllChem.GenerateDepictionMatching2DStructure(mol_json, mol)

molecules=[mol,mol_json]
legend=["Manual SMILE","JSON SMILE"]
Draw.MolsToGridImage(molecules,legends=legend,molsPerRow=2,subImgSize=(250,250))

## Proprietà molecolari di Imatinib

Prima di fare docking, è buona pratica caratterizzare il ligando dal punto di vista chimico-fisico. Il criterio più famoso in drug discovery è la Regola di Lipinski (detta anche *Rule of Five*), una serie di soglie empiriche che predicono se una molecola ha buone probabilità di essere un farmaco orale.

Le quattro proprietà da calcolare sono:

| Proprietà | Soglia Lipinski |
 |------------|------------ |
| Peso molecolare | ≤ 500 Da |
| LogP (lipofilicità)| ≤ 5 |
| Donatori di legami | H≤ 5|
| Accettori di legami |H≤ 10|





In [ ]:
# Usiamo la versione mol

from rdkit.Chem import Descriptors

molwt=Descriptors.MolWt(mol)
molLogP=Descriptors.MolLogP(mol)
numHdon=Descriptors.NumHDonors(mol)
numHacc= Descriptors.NumHAcceptors(mol)


print(f"Molwt: {molwt:.2f}. Regola rispettata: {molwt<=500}")
print(f"MolLogP: {molLogP:.2f}. Regola rispettata: {molLogP<=5}")
print(f"NumHdon: {numHdon}. Regola rispettata: {numHdon<=5}")
print(f"NumHacc: {numHacc}. Regola rispettata: {numHacc<=10}")

## Caricare il target : ABL1

In [ ]:
from dockstring import load_target

In [ ]:
target=load_target("ABL1")

In [ ]:
print(dir(target))

In [ ]:
mol3dpath=target.pdbqt_path
mol3d=open(mol3dpath).read()

In [ ]:
view=py3Dmol.view()
view.addModel(mol3d, "pdbqt")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.show()

# Docking

Normalmente prima del docking, si devono effettuare una serie di passaggi :

- **Preparazione della proteina target**: rimuovere molecole d'acqua, aggiungere gli idrogeni, assegnare le cariche parziali, definire il sito di legame (il "box" di ricerca)

- **Preparazione del ligando** : generare la geometria 3D, assegnare le cariche, definire i legami ruotabili

Vedremo questi step in un tutorial a parte.
In questo possiamo saltarli perché dockstring fa  la parte di preparazione automaticamente .



In [ ]:
# Passiamo a dock lo SMILE dell'Imatinib
score, aux=target.dock(Chem.MolToSmiles(mol))

In [ ]:
print(score)

In [ ]:
print(aux)

In [ ]:
ligand_pdb_string=Chem.MolToPDBBlock(aux['ligand'])

In [ ]:
view=py3Dmol.view()
view.addModel(mol3d, "pdbqt")
view.addModel(ligand_pdb_string, "pdb")
view.setStyle({"model": 0},{"cartoon": {"color": "spectrum"}})
view.setStyle({"model": 1},{"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo({"model":1})
view.show()